# Lab 9 Report: 
## Final Project Codebase

## Project Title: Same as your poster title

### Group Members: Simon Harty 

--------------------

In [19]:
from torchvision import datasets, transforms
from torchvision.models.vision_transformer import EncoderBlock
from torch.utils.data import DataLoader, random_split
import torch
from torch.utils.data import Dataset
from PIL import Image
from pathlib import Path
import pandas as pd
import tqdm

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
CROP_SIZE = 207
INPUT_SIZE = 64
N_WORKERS = 0

DEVICE

device(type='cpu')

## Prepare Data

In [ ]:
#Use one transform using train stats as train and val means / stds are almost the same
transform = transforms.Compose([
    transforms.RandomRotation(degrees=(0, 360)),
    transforms.Normalize(mean=[0.1034, 0.0895, 0.0703], std=[0.1444, 0.1210, 0.1044]),
])

class GalaxyDataset(Dataset):
    def __init__(self, image_paths, labels_csv, transform=None):
        self.transform = transform
        self.labels_df = pd.read_csv(labels_csv, index_col="GalaxyID")
        self.galaxy_ids = [int(p.stem) for p in image_paths]

        pre = transforms.Compose([
            transforms.CenterCrop(CROP_SIZE),
            transforms.Resize(INPUT_SIZE),
            transforms.ToTensor(),
        ])

        self.images = torch.stack([pre(Image.open(p).convert("RGB")) for p in tqdm.tqdm(image_paths)])

    def __getitem__(self, idx):
        img = self.images[idx]
        if self.transform:
            img = self.transform(img)
        label = torch.tensor(self.labels_df.loc[self.galaxy_ids[idx]].values, dtype=torch.float32)
        return img, label
    
    def __len__(self):
        return len(self.images)


data_dir   = Path(r"C:\Users\harty\Code\Galaxy-Classifier\Data\images_training_rev1")
labels_csv = r"C:\Users\harty\Code\Galaxy-Classifier\Data\training_solutions_rev1.csv"
all_paths  = sorted(data_dir.glob("*.jpg"))

In [21]:
indices = torch.randperm(len(all_paths), generator=torch.Generator().manual_seed(42))
n_val   = int(len(all_paths) * 0.1)

train_paths = [all_paths[i] for i in indices[n_val:]]
val_paths   = [all_paths[i] for i in indices[:n_val]]

train_set = GalaxyDataset(train_paths, labels_csv, transform=transform)
val_set   = GalaxyDataset(val_paths, labels_csv, transform=transform)

100%|██████████| 6157/6157 [00:08<00:00, 697.99it/s]


In [22]:
train_mean = train_set.images.mean(dim=[0, 2, 3])
train_std  = train_set.images.std(dim=[0, 2, 3])
print(train_mean, train_std)

val_mean = val_set.images.mean(dim=[0, 2, 3])
val_std  = val_set.images.std(dim=[0, 2, 3])
print(val_mean, val_std)

tensor([0.1034, 0.0895, 0.0703]) tensor([0.1444, 0.1210, 0.1044])
tensor([0.1038, 0.0899, 0.0706]) tensor([0.1442, 0.1209, 0.1044])


## Define Model

In [23]:
import torch
import torch.nn as nn

class ConvBlock(nn.Module):
    """Two conv layers with BN, ReLU, and MaxPool."""
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )

    def forward(self, x):
        return self.block(x)


class ConvTokenizer(nn.Module):
    """Projects feature map into a sequence of tokens for the transformer."""
    def __init__(self, in_channels, embed_dim):
        super().__init__()
        self.proj = nn.Conv2d(in_channels, embed_dim, kernel_size=1)

    def forward(self, x):
        x = self.proj(x)
        B, C, H, W = x.shape
        x = x.flatten(2).transpose(1, 2)
        return x


class CvTBlock(nn.Module):
    """Uses PyTorch's native, optimized Vision Transformer block."""
    def __init__(self, embed_dim, num_heads, mlp_ratio=4.0, dropout=0.1):
        super().__init__()
        
        # PyTorch calculates the hidden MLP dimension directly using mlp_dim
        mlp_dim = int(embed_dim * mlp_ratio)
        
        # Native block combines LayerNorm, MultiheadAttention, and the MLP
        self.transformer_layer = EncoderBlock(
            num_heads=num_heads,
            hidden_dim=embed_dim,
            mlp_dim=mlp_dim,
            dropout=dropout,
            attention_dropout=dropout,
            norm_layer=nn.LayerNorm
        )

    def forward(self, x):
        return self.transformer_layer(x)


class GalaxyClassifier(nn.Module):
    def __init__(
        self,
        in_channels: int = 3,
        num_outputs: int = 37,
        conv_channels: int = 64,
        embed_dim: int = 128,
        num_heads: int = 4,
        num_cvt_layers: int = 2,
        fc_dim: int = 512,
        dropout: float = 0.3,
    ):
        super().__init__()

        # CNN block
        self.conv_block = ConvBlock(in_channels, conv_channels)

        # CvT block
        self.tokenizer = ConvTokenizer(conv_channels, embed_dim)
        self.cvt = nn.Sequential(
            *[CvTBlock(embed_dim, num_heads, dropout=dropout)
              for _ in range(num_cvt_layers)]
        )

        # FC layers
        self.fc = nn.Sequential(
            nn.Linear(embed_dim, fc_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),

            nn.Linear(fc_dim, fc_dim // 2),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),

            nn.Linear(fc_dim // 2, num_outputs),
        )

    def forward(self, x):
        x = self.conv_block(x)
        x = self.tokenizer(x)
        x = self.cvt(x)
        x = x.mean(dim=1)
        x = self.fc(x)  
        return x


model = GalaxyClassifier().to(DEVICE)
dummy = torch.randn(4, 3, 64, 64).to(DEVICE)
out = model(dummy)
print(f"Output shape: {out.shape}")   # expect [4, 37]
print(f"Params: {sum(p.numel() for p in model.parameters()):,}")

Output shape: torch.Size([4, 37])
Params: 650,725


## Define Hyperparameters

In [24]:
BATCH_SIZE = 128
EPOCHS = 16
LR = 1e-3

optimizer = torch.optim.AdamW(model.parameters(), lr = LR)
loss_func = torch.nn.MSELoss()

## Identify Tracked Values

In [25]:
train_losses = []
val_losses = []

## Train Model

In [26]:
pin = DEVICE.type == 'cuda'
train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True,  num_workers=N_WORKERS, pin_memory=pin)
val_loader   = DataLoader(val_set,   batch_size=BATCH_SIZE, shuffle=False, num_workers=N_WORKERS, pin_memory=pin)

import tqdm # Use "for epoch in tqdm.trange(epochs):" to see the progress bar
# Training Loop ---------------------------------------------------------------------------------------
for epoch in tqdm.trange(EPOCHS):
    model.train()
    running_loss = 0

    batch_bar = tqdm.tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}", leave=False)
    for imgs, labels in batch_bar:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = loss_func(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

        batch_bar.set_postfix(loss = f'{loss.item():.4f}')

    train_losses.append(running_loss / len(train_loader))

    # Compute Validation Accuracy ----------------------------------------------------------------------
    model.eval()
    running_loss = 0
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            outputs = model(imgs)
            loss = loss_func(outputs, labels)
            running_loss += loss.item()
            
        # Store validation accuracy for this epoch
        val_losses.append(running_loss / len(val_loader))


TypeError: object of type 'GalaxyDataset' has no len()

## Visualize & Evaluate Model

In [ ]:
import matplotlib.pyplot as plt

plt.plot(train_losses, label="train")
plt.plot(val_losses,   label="val")
plt.xlabel("Epoch")
plt.ylabel("MSE Loss")
plt.legend()
plt.show()